**[Course Home](../README.md) | [Python Lab](../python/04_mutual_information_and_venn_geometry.ipynb) | [Prev: Ch 03: Where Fleet Priors Come From](03_maximum_entropy_priors.ipynb) | [Next: Ch 05: The Geometry of Inference](05_curvature_laplace_and_fisher_information.ipynb)**

*Part II: Multi-Variable Geometry & Causal Architectures — Chapter 04 of 12*

---

# Chapter 04: Shared Outages & Venn Geometry
### *Joint Entropy, Mutual Information, Venn Geometry, & Reversal Paradox*

---

## 1. The Scenario: The 2:00 PM Double Failure

At 2:15 PM, two completely unrelated unit tests fail simultaneously on your CI
dashboard:
1. `test_user_authentication` ($T_1 = 1$)
2. `test_billing_checkout` ($T_2 = 1$)

A junior engineer panics:
> *"The authentication service and the billing service are both broken! Did
> someone push a bug to core business logic?"*

A senior site reliability engineer (SRE) checks the database dashboard and
responds:
> *"Wait. Both of those tests connect to the staging Redis cache ($Z$). At 2:14
> PM, Redis suffered an eviction spike. The tests didn't break each other—Redis
> failed them both."*

We formalize this operational diagnosis as a **3-node Bayesian causal network**
where a hidden infrastructure state $Z \in \{0, 1\}$ acts as a common cause
(confounder) for tests $T_1$ and $T_2$:

```
                      THE CAUSAL FORK ARCHITECTURE
                      
                     +----------------------------+
                     |  Staging Redis Health (Z)  |
                     |  P(Z = 1) = 0.02 (2% down) |
                     +----------------------------+
                                   /  \
                                  /    \
                                 v      v
             +-----------------------+  +-----------------------+
             | Auth Test Status (T1) |  | Billing Test Stat(T2) |
             | P(T1 = 1 | Z)         |  | P(T2 = 1 | Z)         |
             +-----------------------+  +-----------------------+
```

The physical infrastructure parameters are:

$$
P(Z = 1) = 0.02 \quad (\text{2% chance shared database is degraded})
$$

- When infra is healthy ($Z = 0$): tests fail independently at baseline flake
  rate $P(T_i = 1 \mid Z = 0) = 0.01$.
- When infra is degraded ($Z = 1$): both tests fail at high rate
  $P(T_i = 1 \mid Z = 1) = 0.80$.

---

### Step 1: Bayesian Common-Cause Diagnosis

When both $T_1 = 1$ and $T_2 = 1$ fail on the same build, what is the posterior
probability that our shared infrastructure suffered an outage?

We apply Bayes' Theorem. Under conditional independence given $Z$:

$$
P(T_1 = 1, T_2 = 1 \mid Z = 1) = 0.80 \times 0.80 = 0.6400
$$

$$
P(T_1 = 1, T_2 = 1 \mid Z = 0) = 0.01 \times 0.01 = 0.0001
$$

The marginal probability of observing a simultaneous double failure is:

$$
P(T_1 = 1, T_2 = 1) = P(Z = 1) P(T_1 = 1, T_2 = 1 \mid Z = 1)
  + P(Z = 0) P(T_1 = 1, T_2 = 1 \mid Z = 0)
$$

$$
P(T_1 = 1, T_2 = 1) = 0.02(0.64) + 0.98(0.0001) = 0.0128 + 0.000098 = 0.012898
$$

Now compute the posterior probability of a shared infrastructure outage:

$$
P(Z = 1 \mid T_1 = 1, T_2 = 1) = \frac{0.02 \times 0.64}{0.012898}
  = \frac{0.0128}{0.012898} \approx \mathbf{99.24\%}!
$$

Contrast this with observing only a single test failure $T_1 = 1$:

$$
P(T_1 = 1) = 0.02(0.80) + 0.98(0.01) = 0.0160 + 0.0098 = 0.0258
$$

$$
P(Z = 1 \mid T_1 = 1) = \frac{0.02 \times 0.80}{0.0258}
  = \frac{0.0160}{0.0258} \approx 62.02\%
$$

A single failure leaves substantial ambiguity ($37.98\%$ chance the test was
just an isolated baseline flake). But a simultaneous double failure escalates
our outage certainty to **$99.24\%$**!

---

### Step 2: The Information-Theoretic Link

How does this Bayesian causal structure map to Information Theory?

1. **Marginal Mutual Information is Positive ($I(T_1; T_2) > 0$):**
   Marginally (averaging over $Z$), knowing that $T_1 = 1$ provides information
   about $T_2$:

$$
P(T_2 = 1 \mid T_1 = 1) = \frac{P(T_1 = 1, T_2 = 1)}{P(T_1 = 1)}
  = \frac{0.012898}{0.0258} \approx 0.4999 \gg 0.0258
$$

   Because both tests carry information about the hidden common cause $Z$, they
   share positive Mutual Information:

$$
I(T_1; T_2) \approx 0.0481\text{ bits}
$$

2. **Conditional Mutual Information Vanishes ($I(T_1; T_2 \mid Z) = 0$):**
   Once an SRE checks the Redis metrics and confirms $Z = 1$ (or $Z = 0$), the
   tests become strictly conditionally independent:

$$
P(T_1, T_2 \mid Z) = P(T_1 \mid Z) P(T_2 \mid Z)
$$

   The conditional mutual information collapses to exactly zero:

$$
I(T_1; T_2 \mid Z) = 0.0000\text{ bits}
$$

   Conditioning on the common cause **destroys the apparent correlation**!

---

## 2. The Linear Blind Spot of Classical Correlation

For more than a century, classical statistics measured the relationship between
two variables using the **Pearson correlation coefficient** $r \in [-1, 1]$:

$$
r = \frac{\text{Cov}(X, Y)}{\sigma_X \sigma_Y}
$$

Pearson correlation has a fatal limitation: **it measures exclusively linear
(straight-line) relationships**.

Consider an intermittent hardware race condition where sensor reading $Y$ is a
deterministic quadratic function of CPU load $X \sim \mathcal{U}[-1, 1]$:

$$
Y = X^2
$$

Because $X$ is symmetric around zero:

$$
\mathbb{E}[X] = 0, \quad \mathbb{E}[X^3] = 0 \implies
\text{Cov}(X, Y) = \mathbb{E}[X^3] - \mathbb{E}[X]\mathbb{E}[Y] = 0
$$

$$
\mathbf{r = 0.0000}
$$

A standard linear analysis concludes that $X$ and $Y$ are completely
unrelated! Yet in physical reality, $Y$ is **100% determined by $X$**. If you
know $X = 0.7$, you know $Y = 0.49$ with absolute certainty.

Information Theory provides the universal, non-linear, non-parametric metric
of dependence: **Mutual Information $I(X; Y)$**.

---

## 3. Mutual Information & The Information Venn Diagram

Before analyzing two variables together, we define their **Joint Entropy**
$H(X, Y)$:

$$
\mathbf{H(X, Y) \equiv -\sum_{x, y} P(x, y) \log_2 P(x, y) \quad [\text{bits}]}
$$

Joint entropy represents the **total union** of information in the system—the
number of bits needed to store both outcomes simultaneously.

### The Overlap: Mutual Information $I(X; Y)$
The amount of information shared between $X$ and $Y$ is the **Mutual
Information**:

$$
\mathbf{I(X; Y) \equiv}
\sum_{x, y} P(x, y) \log_2 \frac{P(x, y)}{P(x)P(y)} \quad [\text{bits}]
$$

* If $X$ and $Y$ are strictly independent:
  $P(x, y) = P(x)P(y) \implies \log_2(1) = 0 \implies \mathbf{I(X; Y) = 0}$.
* If $X$ and $Y$ are identical twins ($Y = X$):
  $\mathbf{I(X; Y) = H(X)}$. Knowing $X$ completely resolves $Y$.

---

### The Information Venn Partition

In 1991, Raymond Yeung proved that Shannon information measures form a
**signed measure** over the space of random variables. Information obeys the
exact geometry of a Venn diagram:

```
                THE INFORMATION VENN PARTITION
                
<------------- H(T1) = 0.1729 bits ------------->
             (Total Auth Uncertainty)
                         <------------- H(T2) = 0.1729 bits ------------->
                                      (Total Billing Uncertainty)
+-----------------------+-----------------------+-----------------------+
|                       |                       |                       |
|       H(T1 | T2)      |       I(T1 ; T2)      |       H(T2 | T1)      |
|      0.1248 bits      |      0.0481 bits      |      0.1248 bits      |
|                       |                       |                       |
|   Residual Auth Risk  |  Shared Outage Signal | Residual Billing Risk |
|  (After checking T2)  |   (Redis Information) |  (After checking T1)  |
|                       |                       |                       |
+-----------------------+-----------------------+-----------------------+
<-------------------------- H(T1, T2) = 0.2977 bits ------------------->
                           The Total Joint Union
```

From this geometric duality, every fundamental identity can be read off by eye:
1. **The Whole is the Sum of Disjoint Partitions**:

$$
H(T_1, T_2) = H(T_1 \mid T_2) + I(T_1; T_2) + H(T_2 \mid T_1)
$$

2. **Mutual Information is the Intersection**:

$$
I(T_1; T_2) = H(T_1) + H(T_2) - H(T_1, T_2)
$$

$$
I(T_1; T_2) = H(T_1) - H(T_1 \mid T_2) = H(T_2) - H(T_2 \mid T_1)
$$

3. **Variation of Information is the Symmetric Difference**:

$$
\mathbf{VI(T_1; T_2) \equiv H(T_1, T_2) - I(T_1; T_2)}
  = H(T_1 \mid T_2) + H(T_2 \mid T_1) = 0.2496\text{ bits}
$$

*(Unlike mutual information, $VI(T_1; T_2)$ is a true mathematical distance
metric: it satisfies non-negativity, symmetry, and the triangle inequality
$VI(X; Z) \le VI(X; Y) + VI(Y; Z)$).*

---

## 4. The Causal Architectures: Confounders vs. Colliders (The Reversal Paradox)

In marginal entropy, **conditioning never increases uncertainty on average**:

$$
H(X \mid Z) \le H(X)
$$

*"Information can never hurt."*

It is tempting to assume by false analogy that conditioning on a third variable
$Z$ should also reduce or preserve mutual information:

$$
I(X; Y \mid Z) \stackrel{?}{\le} I(X; Y) \quad \text{--- DANGEROUS FALLACY!}
$$

**Conditioning on a third variable $Z$ can dramatically increase mutual
information!**

To understand why, we compare two fundamental causal architectures: **The Fork
(Confounder)** versus **The Collider (Berkson's Bias / XOR)**.

---

### Architecture A: The Fork / Confounder ($T_1 \leftarrow Z \to T_2$)
In our 2:15 PM outage scenario:
* $Z$: Staging Redis cache health ($0 = \text{Healthy}, 1 = \text{Down}$).
* $T_1$: Auth test failure.
* $T_2$: Billing test failure.

Because both tests depend on Redis, they exhibit positive mutual information:

$$
I(T_1; T_2) \approx 0.0481\text{ bits} > 0
$$

To a naive observer, auth failures seem to predict billing failures!

However, once an SRE logs into Redis and conditions on $Z$:

$$
I(T_1; T_2 \mid Z) = 0.0000\text{ bits}
$$

**Conditioning on the common cause destroys the mutual information.** The
correlation was completely spurious, mediated through $Z$.

---

### Architecture B: The Collider / Berkson's Bias ($X \to Z \leftarrow Y$)
Now consider two completely independent services:
* $X$: Payment gateway timeout ($50/50$ random flake).
* $Y$: Email worker thread crash ($50/50$ random flake).

Because they are independent:

$$
I(X; Y) = 0.0000\text{ bits}
$$

Now suppose your CI runner has a shared queue limit $Z$: the build fails if
**either $X$ fails OR $Y$ fails (or both)**:

$$
Z = X \text{ OR } Y
$$

Now, condition on observing a build failure ($Z = 1$):
* Suppose you inspect Worker 1 and find that the payment gateway was
  completely fine ($X = 0$).
* You ask: *"Wait, how did the build fail if the payment gateway was healthy?"*
* Because $(0, 0)$ is impossible under $Z = 1$, **the email worker must have
  crashed ($Y = 1$) with 100% certainty!**

$$
P(Y = 1 \mid X = 0, Z = 1) = 1.00 \quad (100\%)
$$

Knowing $X$ now completely determines $Y$!

$$
\mathbf{I(X; Y \mid Z = 1) > 0}
$$

> **The Reversal Paradox**: Conditioning on a common effect (a collider)
> **creates mutual information out of thin air**. In statistical systems, this
> is called **"Explaining Away"** or **Berkson's Paradox**.

---

## 5. Interaction Information: Synergy vs. Redundancy

To formalize this mathematically, we define **3-Way Interaction Information**:

$$
\mathbf{I(X; Y; Z) \equiv I(X; Y) - I(X; Y \mid Z)}
$$

* **If $I(X; Y; Z) > 0$ (Redundancy / The Fork)**:
  Conditioning on $Z$ reduced mutual information ($I(X;Y \mid Z) < I(X;Y)$).
  $Z$ was a confounder explaining shared variance. In our Redis scenario:
  $$I(T_1; T_2; Z) = 0.0481 - 0.0000 = +0.0481\text{ bits} > 0$$
* **If $I(X; Y; Z) < 0$ (Synergy / The Collider)**:
  Conditioning on $Z$ increased mutual information ($I(X;Y \mid Z) > I(X;Y)$).
  $X$ and $Y$ are synergistic puzzle pieces that only make sense together!
  In the OR alert scenario:
  $$I(X; Y; Z) = 0.0000 - 0.3113 = -0.3113\text{ bits} < 0$$

---

## 6. Nuisance Entropy: The Secret to Machine Learning Generalization

This brings us to a fundamental insight in modern deep learning and systems
diagnosis.

Suppose an autonomous triage bot ingests build logs $L$ to classify the root
cause of an outage $Y$. The total mutual information between the raw logs and
the bot's hidden state $H$ decomposes as:

$$
\mathbf{I(\text{Logs}; H) = \underbrace{I(Y; H)}_{\text{Signal}}
  + \underbrace{I(\text{Logs}; H \mid Y)}_{\text{Nuisance Noise}}}
$$

* **$I(Y; H)$**: The true causal defect (e.g. database deadlock).
* **$I(\text{Logs}; H \mid Y)$**: Nuisance variation—worker hostnames,
  microsecond timestamps, container IP addresses, ambient CPU loads.

A naive neural network memorizes hostnames and timestamps (overfitting). A
robust, generalizable triage bot acts as a **nuisance entropy filter**: it
ruthlessly drives $I(\text{Logs}; H \mid Y) \to 0$, compressing raw log noise
onto a clean, invariant manifold where root causes can be classified
effortlessly.

---

## 7. Summary: The Golden Rules of Chapter 04

1. **Mutual Information captures all dependencies**: Unlike Pearson correlation
   $r$, $I(X; Y)$ detects quadratic, non-linear, and logical relationships.
2. **The Information Venn Diagram maps to set theory**:
   * Total system uncertainty = Set Union $H(T_1, T_2) = 0.2977\text{ bits}$
   * Shared diagnostic information:
     Set Intersection $I(T_1; T_2) = 0.0481\text{ bits}$$
   * Residual doubt = Set Difference $H(T_1 \mid T_2) = 0.1248\text{ bits}$
   * Variation of Information:
     Symmetric Difference $VI(T_1; T_2) = 0.2496\text{ bits}$$
3. **The Fork vs. Collider Rule**:
   * Conditioning on a common cause (Fork) **destroys** mutual information
     ($I(T_1; T_2 \mid Z) < I(T_1; T_2)$).
   * Conditioning on a common effect (Collider) **creates** mutual information
     ($I(X; Y \mid Z) > I(X; Y)$).
4. **Bayesian Diagnosis Explains the Venn Overlap**: Simultaneous failures
   drastically escalate the posterior probability of a shared infrastructure
   outage ($99.24\%$ vs. $62.02\%$ for a single failure) precisely because the
   tests share mutual information mediated through $Z$.
5. **Negative Interaction Information indicates synergy**: $I(X; Y; Z) < 0$ is
   the mathematical signature of colliders and synergistic interactions.

Now, open the companion Python Lab to simulate the 2:15 PM double failure,
verify the Venn partitions numerically, and witness the Reversal Paradox in
code!

---

**[Course Home](../README.md) | [Python Lab](../python/04_mutual_information_and_venn_geometry.ipynb) | [Prev: Ch 03: Where Fleet Priors Come From](03_maximum_entropy_priors.ipynb) | [Next: Ch 05: The Geometry of Inference](05_curvature_laplace_and_fisher_information.ipynb)**
